# పాఠం 13 - కాగ్ణీ తెలిసికొనటి గ్రాఫ్‌లతో ఏజెంట్ స్మృతి


## సెటప్

ఈ నోటుబుక్ ద్వారా మీరు [**Cognee**](https://www.cognee.ai/) నాలెడ్జ్ గ్రాఫ్‌లు మరియు **Microsoft Agent Framework** (MAF) ఉపయోగిస్తూ స్థిరమైన మెమరీతో కూడిన మేధావి **కోడింగ్ అసిస్టెంట్ని** ఎలా నిర్మించాలో తెలుసుకోవచ్చు.

Cognee నిర్మంజలంగా ఉండే టెక్స్ట్‌ను వెక్టర్ ఎంబెడింగ్స్ మద్దతుతో కూడిన, క్వెరీ చేయదగిన నాలెడ్జ్ గ్రాఫ్‌గా మార్చుతుంది — ఈ విధంగా మీ ఏజెంట్‌కి సమృద్ధిగా, సంబంధాన్ని గుర్తించే దీర్ఘకాలిక మెమరీ లభిస్తుంది.

### మీరు ఏమి నేర్చుకుంటారు
1. **నాలెడ్జ్ గ్రాఫ్‌లను నిర్మించండి**: డెవలపర్ ప్రొఫైల్‌లు మరియు ఉత్తమ పద్ధతులను నిర్మంజల, క్వెరీ చేయదగిన నాలెడ్జ్‌గా మార్చడం.
2. **Cogneeని MAFతో సమగ్రపంచుకోండి**: MAF ఏజెంట్ Cognee నాలెడ్జ్ గ్రాఫ్‌ను క్వెరీ చేయడానికి `@tool` ఫంక్షన్లను ఉపయోగించడం.
3. **సెషన్-సూచక సంభాషణలు**: ఒకే సెషన్‌లో గల అనేక ప్రశ్నలకు మధ్య సందర్భాన్ని నిర్వహించడం.
4. **దీర్ఘకాలిక మెమరీ**: కీలకమైన నాలెడ్జ్‌ను సెషన్‌ల మధ్య నిలబెడుతూ కొత్త సంభాషణల్లో పొందడం.

### అవసరాలు
- Python 3.9+
- సెస్‌షన్ నిర్వహణ కోసం స్థానికంగా Redis నడుపుట (`docker run -d -p 6379:6379 redis`)
- LLM API కీ (ఉదా: OpenAI) — `.env` లో `LLM_API_KEY` సెట్ చేయాలి
- `.env` లో `CACHING=true` (Cognee సెషన్‌లకై అవసరం)
- డిప్లాయ్ చేసిన చాట్ మోడల్‌తో Azure AI Foundry ప్రాజెక్టు
- `.env` లో `AZURE_AI_PROJECT_ENDPOINT` మరియు `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI లో లాగిన్ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ఏజెంట్ మెమరీ తరజాతులు

ఈ నోట్బుక్ ప్రధాన పాఠం 13 నోట్బుక్ నుండి అదే మూడువిధమైన మెమరీలను పరిశీలిస్తుంది, కానీ దీర్ఘకాలిక మెమరీ బ్యాక్‌ఎండ్‌గా Cogneeని ఉపయోగిస్తుంది:

| మెమరీ తరహా | యాంత్రిక విధానం | జీవితకాలం |
|---|---|---|
| **వర్కింగ్** | `agent.create_session()` (MAF) | ఒక్కొక్క సంభాషణ |
| **షార్ట్-టర్మ్** | Cognee సెషన్ క్యాష్ (Redis) | ఒక్కొక్క సెషన్ |
| **లాంగ్-టర్మ్** | Cognee జ్ఞాన గ్రాఫ్ + వెక్టర్లు | శాశ్వతం |

### Cognee యొక్క మెమరీ నిర్మాణం
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## కాగ్నీ స్టోరేజ్ సిద్ధం చేయండి


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## భాగం 1 — జ్ఞానపు మూలాన్ని నిర్మించడం

మేము మా కోడింగ్ సహాయకుడికై సమగ్ర జ్ఞానపు మూలాన్ని సృష్టించడానికి మూడు రకాల డేటాను తీసుకుంటాము:

1. **డెవలపర్ ప్రొఫైల్** — వ్యక్తిగత నైపుణ్యం మరియు సాంకేతిక నేపథ్యం  
2. **Python ఉత్తమ ఆచరణలు** — ప్రాక్టికల్ మార్గదర్శకాలతో Python యొక్క జెన్  
3. **చారిత్రక సంభాషణలు** — డెవలపర్లు మరియు AI సహాయకుల మధ్య గత ప్రశ్నోత్తర సెషన్లు


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## జ్ఞాన గ్రాఫ్‌ను దృష్టిలో ఉంచండి

Cognee అది తీసుకున్న అంశాలు మరియు సంబంధాల యొక్క ఇంటరాక్టివ్ HTML విజువలైజేషన్‌ను రూపొందించగలదు.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memifyతో మెమరీని సమృద్ధిగా చేయండి

`memify()` జ్ఞాన గ్రాఫ్‌ని విశ్లేషించి, తెలివైన నియమాలను సృష్టిస్తుంది — నమూనాలు, ఉత్తమ ఆచారాలను, మరియు భావాల మధ్య సంబంధాలను గుర్తిస్తుంది.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## భాగం 2 — Cognee టూల్స్‌తో MAF ఏజెంట్

ఇప్పుడు మేము Cognee యొక్క జ్ఞాన గ్రాఫ్‌ను `@tool` ఫంక్షన్‌లు ద్వారా విచారించగల MAF ఏజెంట్‌ను సృష్టిస్తాము. ఇది ఏజెంట్‌కు సెషన్ల ద్వారా సంభాషణ సందర్భాన్ని պահպանిస్తూ గ్రాఫ్-జ్ఞానాసక్తమైన సేమాంటిక్ సెర్చ్ యొక్క పూర్తి శక్తిని ఉపయోగించడానికి అనుమతిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## సెషన్లతో పని జ్ఞాపకం

`AgentSession` (`agent.create_session()` ద్వారా సృష్టించబడింది) ఒక సెషన్‌లో పని జ్ఞాపకాన్ని అందిస్తుంది. ఏజెంట్ ముందస్తు సందేశాలను సూచించవచ్చు మరియు అలాగే Cognee యొక్క దీర్ఘకాలిక జ్ఞాన గ్రాఫ్‌ను కూడా విచారించవచ్చు.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## కొత్త సెషన్ — దీర్ఘకాలిక జ్ఞాపకం కొనసాగుతుంది

కొత్త సెషన్‌ను ప్రారంభించడం వర్కింగ్ మెమరీని శుభ్రం చేస్తుంది, కానీ Cognee జ్ఞాన గ్రాఫ్ ఇంకా అందుబాటులో ఉంటుంది. ఏజెంట్ పూర్తిగా కొత్త సంభాషణలో అదే దీర్ఘకాలిక జ్ఞానాన్ని తిరిగి పొందగలడు.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## సారాంశం

ఈ నోట్బుక్‌లో మీరు **MAF యొక్క వర్కింగ్ మేమరీ** (`agent.create_session()`)ను **Cognee యొక్క దీర్ఘకాలిక జ్ఞాన గ్రాఫ్**తో కలిపి ఒక కోడింగ్ అసిస్టెంట్‌ను నిర్మించారు.

### మీరు నేర్చుకున్నది
1. **జ్ఞాన గ్రాఫ్ నిర్మాణం**: Cognee అసంఘటిత టెక్స్ట్‌ను శోషించి గ్రాఫ్ + వెక్టర్ మేమరీని నిర్మిస్తుంది.
2. **memify తో గ్రాఫ్ పొడగింపు**: మీ ఉన్న గ్రాఫ్ పై ఆవిర్భవించిన నిజాలు మరియు సమృద్ధి సంబంధాలు.
3. **MAF + Cognee సమ్మిళితం**: `@tool` ఫంక్షన్లు MAF ఏజెంట్ Cognee గ్రాఫ్‌ను సహజంగానే క్వెరీ చేయడానికి అనుమతిస్తాయి.
4. **వర్కింగ్ మేమరీ + దీర్ఘకాలిక మేమరీ**: `AgentSession` (`agent.create_session()` ద్వారా) సెషన్ సందర్భాన్ని అందిస్తుంది, Cognee 지속적인 జ్ఞానాన్ని అందిస్తుంది.
5. **NodeSets తో వడపోత శోధన**: జ్ఞాన గ్రాఫ్ యొక్క నిర్దిష్ట ఉపసమూహాలను లక్ష్యం చేయండి (ఉదా: కేవలం సిద్ధాంతాలు).

### ముఖ్యమైన పాఠాలు
- **Cognee** రా టెక్స్‌ను నిర్మిత, సంబంధ-అవగాహన మేమరీగా మార్చుతుంది — సాదారణ వెక్టర్ స్టోర్ కంటే మరింత శక్తివంతం.
- **`@tool` ఫంక్షన్లు** MAF ఏజెంట్లను బాహ్య జ్ఞాన వ్యవస్థలతో శుభ్రంగా జాడుతాయి.
- **`AgentSession`** (`agent.create_session()` ద్వారా) per-కళెక్టర్ సందర్భాన్ని దీర్ఘకాలిక జ్ఞానంతో వేరు ఉంచుతుంది.
- అదే జ్ఞాన గ్రాఫ్ అనేక సెషన్లు మరియు ఏజెంట్లకు సేవలందిస్తుంది.

### వాస్తవ ప్రపంచ దరఖాస్తులు
- **డెవలపర్ కోపైలాట్‌లు**: కోడ్ సమీక్ష, సంఘటన విశ్లేషణ, ఆర్కిటెక్చర్ సహాయకులు
- **కస్టమర్-ముఖంగా కోపైలాట్‌లు**: ఉత్పత్తి డాక్యుమెంట్స్, FAQs మరియు CRM నోట్లపై సపోర్ట్ ఏజెంట్లు
- **అంతర్గత నిపుణ కోపైలాట్‌లు**: విధానాలు, చట్టాలు, లేదా భద్రత సహాయకులు మార్గదర్శకాలు ఆధారంగా తర్కం చేస్తారు
- **షేర్డ్ డేటా లేయర్లు**: నిర్మిత మరియు అసంఘటిత డేటాను ఒకే క్వెరీ చేయగలిగే గ్రాఫ్‌గా సంయోజింపజేయండి

### తదుపరి దశలు
- Cognee లో కాలానుగుణ అవగాహనతో ప్రయోగాలు చేయండి
- డొమైన్-ప్రత్యేక గ్రాఫ్ నాణ్యత కోసం OWL ఆంటాలజీ నిర్వచించండి
- సమయం పొడిగితే రిక్రూట్ మెరుగుపర్చడానికి వినియోగదారు అభిప్రాయం లూప్‌లను జోడించండి
- అదే Cognee మేమరీ లేయర్ ను పంచుకునే బహుళ ఏజెంట్ వ్యవస్థలకు స్కేల్ చేయండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వం కోసం ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో తప్పు లేదా తప్పుదోవలు ఉండొచ్చు. అసలు పత్రం దాని స్థానిక భాషలో ఉన్నదిగా మాత్రమే అధికారిక మూలం గా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం సూచించబడుతుంది. ఈ అనువాదం వాడకం వల్ల ఎదురయ్యే ఏ భ్రమలు లేదా తప్పు అర్థం చేసుకునే పరిస్థితులకు మేము బాధ్యులను కావాలెను.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
